# FSL-SAGE on Colab GPU

Clones this repo from GitHub, installs dependencies, and runs a quick GPU smoke test
(see `docs/part0-mnist-smoke-test.md` for the equivalent CPU run this mirrors).

**Before running:** `Runtime -> Change runtime type -> T4 GPU` (or better).

Datasets (`datas/`) and run outputs (`saves/`) are stored on your Google Drive so they
persist across Colab session resets instead of re-downloading/re-running every time.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Change BRANCH once this work lands on master.
REPO_URL = "https://github.com/juniorfelix998/FSL-SAGE.git"
BRANCH = "ft/add-mnist"

WORKSPACE = "/content/drive/MyDrive/fsl-sage-colab"
REPO_DIR = f"{WORKSPACE}/FSL-SAGE"

import os
os.makedirs(WORKSPACE, exist_ok=True)

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch origin {BRANCH}
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

In [ ]:
# src/main.py resolves datasets/saves as '../datas' and '../saves' relative to
# src/, so point those at persistent Drive folders instead of Colab's ephemeral disk.
DRIVE_DATAS = f"{WORKSPACE}/datas"
DRIVE_SAVES = f"{WORKSPACE}/saves"
os.makedirs(DRIVE_DATAS, exist_ok=True)
os.makedirs(DRIVE_SAVES, exist_ok=True)

for name, target in (("datas", DRIVE_DATAS), ("saves", DRIVE_SAVES)):
    link = f"{REPO_DIR}/{name}"
    if os.path.islink(link) or os.path.exists(link):
        continue
    os.symlink(target, link)

In [ ]:
# torch/torchvision are pinned to Colab's OWN already-installed versions (not the
# repo's local conda_env.yaml pin of torch==2.5.1) so pip has no reason to touch them --
# swapping torch pulls in a different CUDA toolkit than the one Colab's preinstalled
# RAPIDS stack (cuml/cudf/libraft/libcuvs/cuda-python) was built against, which is what
# caused the wall of "cuda-toolkit ... incompatible" resolver errors. requests is bumped
# to 2.32.4 to match what google-colab/google-adk already require, for the same reason.
# Check !python -c "import torch, torchvision; print(torch.__version__, torchvision.__version__)"
# on a fresh runtime if these ever drift from what Colab ships.
!pip install -q torch==2.13.0 torchvision==0.28.0 hydra-core==1.3.2 hydra-joblib-launcher==1.2.0 \
  omegaconf==2.3.0 wandb==0.19.3 numpy==2.1.3 pandas==2.2.3 scipy==1.14.1 matplotlib==3.9.2 \
  h5py==3.12.1 pyyaml==6.0.2 tqdm==4.67.0 requests==2.32.4 pillow==11.0.0 prettytable==3.12.0 \
  joblib==1.4.2 antlr4-python3-runtime==4.9.3 gitpython==3.1.43

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds=3 save=False device=cuda model=resnet18 dataset=mnist

## Running a single method standalone

Supported `algorithm=` keys: `fed_avg`, `sl_multi_server` (SplitFedv1),
`sl_single_server` (SplitFedv2), `cse_fsl`, `fsl_sage`, `ho_sfl`, `mu_splitfed`, `dsl_aux`.

**`mu_splitfed` provenance note:** this is HKU-WILL-Lab/HO-SFL's own third-party
CV/ResNet18 reimplementation of MU-SplitFed, not the original Johnny-Zip/MU-SplitFed
authors' code (their published repo is LLM-only and non-functional as published --
see `src/algos/mu_splitfed.py` for details). Treat any MU-SplitFed numbers accordingly.

Each cell below runs one method for `rounds=3` (a quick sanity check, not a real
result) so you can test any single method on its own without running the full sweep.

**`dsl_aux` provenance note:** AI-assisted no-code reimplementation of DSL-Aux (arXiv:2601.19261), adapted from a partial third-party reference (juniorfelix998/sl-fl-dgl) -- not validated against the paper's own reported numbers. It is also non-federated (single client/server split, no weight aggregation); `num_clients=1` is its paper-faithful setting, though the cell below runs it with the notebook's usual default like every other method for consistency.

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds=3 save=False device=cuda model=resnet18 dataset=mnist algorithm=fed_avg

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds=3 save=False device=cuda model=resnet18 dataset=mnist algorithm=sl_multi_server

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds=3 save=False device=cuda model=resnet18 dataset=mnist algorithm=sl_single_server

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds=3 save=False device=cuda model=resnet18 dataset=mnist algorithm=cse_fsl

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds=3 save=False device=cuda model=resnet18 dataset=mnist algorithm=fsl_sage

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds=3 save=False device=cuda model=resnet18 dataset=mnist algorithm=ho_sfl

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds=3 save=False device=cuda model=resnet18 dataset=mnist algorithm=mu_splitfed

In [ ]:
%cd {REPO_DIR}/src
!python main.py rounds=3 save=False device=cuda model=resnet18 dataset=mnist algorithm=dsl_aux

## Running everything: sweep + measurement table + plots in one command

`run_mnist_benchmark.py` is the "one main" entry point: it sweeps all 7 tabled
methods (SplitFedv1, SplitFedv2, CSE-FSL, FSL-SAGE, HO-SFL, MU-SplitFed,
DSL-Aux) across both MNIST distributions (IID and Dirichlet alpha=0.5),
builds the measurement table (comm cut/weights/total, cut vs. weights
share, accuracy, latency, peak memory),
and generates accuracy/communication-load plots -- all from a single command.

**At `--rounds 3` (the default below) this is a pipeline/plumbing check, not a
reportable result** -- it confirms every method runs end-to-end and produces a
`results.json` the table/plot code can parse, not real accuracy numbers. For real
numbers, raise `--rounds` (README/`config.yaml` default is 200) and, per
`CLAUDE.md`'s "Seeds: 3 default", repeat across 3 seeds. At that scale a single
free-tier Colab session likely won't finish in one sitting -- use `--methods` to
re-invoke this for a subset of methods across multiple sessions; each sweep run is
additive (new timestamped folders under `saves/`), nothing gets overwritten.

In [ ]:
%cd {REPO_DIR}/inference
!python run_mnist_benchmark.py --rounds 3 --seed 200 --device cuda

In [ ]:
# Inline display of the table + plots, so results are visible without downloading
# anything from Drive.
import glob
from IPython.display import Image, display

print(open(f"{REPO_DIR}/inference/benchmark_table_mnist.txt").read())

for png in sorted(glob.glob(f"{REPO_DIR}/inference/plots_mnist/**/*.png", recursive=True)):
    print(png)
    display(Image(filename=png))

## Running a custom experiment

Use the README's Hydra override syntax to run any method/model/dataset combination,
e.g.:

```python
!python main.py algorithm=fsl_sage model=resnet18 dataset=cifar10 \\
  dataset.distribution=noniid_dirichlet dataset.alpha=0.5 save=False device=cuda
```

Results land under
`saves/<algorithm>/<model>/<dataset>-<distribution>/.../results.json`, which (via
the symlink above) is actually on your Drive at
`fsl-sage-colab/saves/...` so it survives runtime resets.